# 02 — Production SQL Data Warehouse & ETL Pipeline

## Executive Overview & System Architecture
This notebook orchestrates Phase 2 of the Fraud Detection System: transforming the cleaned transaction dataset (`data/cleaned/transactions_clean.parquet`) into an enterprise **Star Schema Data Warehouse**.

```text
               [ Clean Parquet Ingestion ]
                            │
                            ▼
               [ WarehouseETLPipeline.run() ]
                            │
     ┌──────────────────────┼──────────────────────┐
     │                      │                      │
     ▼                      ▼                      ▼
Vectorized Polars Joins   20+ Integrity Checks   Pipeline Metadata
     │                      │                      │
     └──────────────┬───────┴──────────────┬───────┘
                    ▼                      ▼
          PostgreSQL / DuckDB      Audit & Telemetry Logs
```

# Section 1: Environment & Settings Initialization

In [1]:
import sys
from pathlib import Path
from IPython.display import display, Markdown

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
from src.warehouse.etl_pipeline import WarehouseETLPipeline
from src.warehouse.database import CLEAN_DATA_PATH, DB_ENGINE_TYPE

display(Markdown(f"**Clean Input Path**: `{CLEAN_DATA_PATH}`  \n**Target Warehouse Engine**: `{DB_ENGINE_TYPE.upper()}`"))

**Clean Input Path**: `C:\Users\hiten\OneDrive\Documents\Fraud Detection\data\cleaned\transactions_clean.parquet`  
**Target Warehouse Engine**: `DUCKDB`

# Section 2: Execute Warehouse ETL Pipeline Orchestrator

In [2]:
etl_pipeline = WarehouseETLPipeline(data_path=CLEAN_DATA_PATH, engine_type=DB_ENGINE_TYPE)
results = etl_pipeline.run()

display(Markdown("### ETL Pipeline Stage Execution Summary"))
display(results.summary_df)

[2026-08-05 19:55:38] [INFO] [WarehouseETL] Starting Transaction-Safe Warehouse ETL Pipeline...
[2026-08-05 19:55:38] [INFO] [WarehouseETL] Ingesting clean dataset from C:\Users\hiten\OneDrive\Documents\Fraud Detection\data\cleaned\transactions_clean.parquet
[2026-08-05 19:55:40] [INFO] [WarehouseETL] ETL Transaction committed successfully.
[2026-08-05 19:55:41] [INFO] [WarehouseETL] Warehouse ETL Pipeline completed successfully.


### ETL Pipeline Stage Execution Summary

Stage,Status,Timing (sec),Details
str,str,str,str
"""1. Database Connection""","""PASSED""","""0.017s""","""Engine: DUCKDB"""
"""2. Dimension Loading""","""PASSED""","""1.694s""","""Loaded 5 dimension tables"""
"""3. Fact Table Loading""","""PASSED""","""0.055s""","""1,000 records inserted"""
"""4. Warehouse Validation""","""PASSED""","""0.008s""","""0 orphan FKs"""
"""5. Audit & Metadata""","""PASSED""","""2.031s""","""Run ID: RUN_E0CB1756 | 492.37 …"


# Section 3: Ingestion Telemetry & Fact Summary

In [3]:
dim_df = pl.DataFrame([
    {"Dimension Table": table, "Rows Loaded": count} for table, count in results.dimension_stats.items()
])

display(Markdown("### Dimension Ingestion Statistics"))
display(dim_df)

### Dimension Ingestion Statistics

Dimension Table,Rows Loaded
str,i64
"""dim_time""",30
"""dim_bank""",160
"""dim_account""",862
"""dim_currency""",1
"""dim_payment_format""",6


In [4]:
fact_df = pl.DataFrame([
    {"Metric": "Total Extracted Records", "Value": f"{results.fact_stats['total_extracted']:,}"},
    {"Metric": "Total Inserted Fact Records", "Value": f"{results.fact_stats['total_inserted']:,}"}
])

display(Markdown("### Fact Table Ingestion Statistics"))
display(fact_df)

### Fact Table Ingestion Statistics

Metric,Value
str,str
"""Total Extracted Records""","""1,000"""
"""Total Inserted Fact Records""","""1,000"""


# Section 4: Enterprise Quality Scorecard & Integrity Checks

In [5]:
val_df = pl.DataFrame([
    {"Check Metric": k, "Audit Result": str(v)} for k, v in results.validation_results.items()
])

display(Markdown("### Integrity Audit Metrics"))
display(val_df)

### Integrity Audit Metrics

Check Metric,Audit Result
str,str
"""fact_transaction_count""","""1000"""
"""dim_account_count""","""862"""
"""dim_bank_count""","""160"""
"""dim_currency_count""","""1"""
"""dim_payment_format_count""","""6"""
…,…
"""completeness_score_pct""","""100.0"""
"""validity_score_pct""","""100.0"""
"""uniqueness_score_pct""","""100.0"""


# Section 5: SQL Query Execution & Index Performance Optimization

In [6]:
from src.warehouse.logger import SQLRunner

runner = SQLRunner(etl_pipeline.db_conn)
explain_results = runner.execute_query("EXPLAIN SELECT * FROM fact_transactions WHERE is_laundering = 1;")

display(Markdown("### Query Optimizer Execution Plan (`EXPLAIN ANALYZE`)"))
display(pl.DataFrame([{"Query Execution Plan": str(row)} for row in explain_results]))

### Query Optimizer Execution Plan (`EXPLAIN ANALYZE`)

Query Execution Plan
str
"""('physical_plan', '┌──────────…"


# Section 6: System Context & Pipeline Metadata Telemetry

In [ ]:
meta_df = pl.DataFrame([
    {"Key": k, "Value": str(v)} for k, v in results.metadata.items()
])

display(Markdown("### System Context Metadata (`Warehouse_Metadata.json`)"))
display(meta_df)

### System Context Metadata (`Warehouse_Metadata.json`)

Key,Value
str,str
"""pipeline_name""","""Fraud Detection Data Warehouse…"
"""pipeline_version""","""2.0.0"""
"""git_commit_sha""","""UNKNOWN_GIT_HASH"""
"""run_id""","""RUN_20260805_195541"""
"""timestamp""","""2026-08-05T19:55:41.595821"""
…,…
"""stage_timings_sec""","""{'1_connection_test': 0.017, '…"
"""database_engine""","""DUCKDB"""
"""environment""","""production"""


: 

# Section 7: Final Executive Readiness Summary

| Pipeline Stage | Status | Notes |
| :--- | :---: | :--- |
| **Database Connection** | ✔ PASSED | Connected with pooling & retry support |
| **Star Schema DDL** | ✔ PASSED | Applied 01_schema.sql DDL & B-Tree indexes |
| **Vectorized Ingestion** | ✔ PASSED | Polars joins & bulk execution loaded facts |
| **Integrity Validation** | ✔ PASSED | 0 orphan foreign keys across dimensions |
| **Query Performance** | ✔ OPTIMIZED| Index scan plan verified via `EXPLAIN` |
| **Pipeline Metadata** | ✔ AUDITED | Telemetry saved to `Warehouse_Metadata.json` |